In [19]:
import torch
from pathlib import Path

In [20]:
# Get checkpoints files from the range 
ckpt_dir = Path("/data/ratna/aug/blur_weak/eval") # RSG - changed from /data/OM_checkpoints , /data/ratna_retrain/eval
steps = list(range(87500, 137501, 12500))  # RSG - change this for a new range
ckpt_paths = [ ckpt_dir / f"training_{step}" / "teacher_checkpoint.pth" for step in steps]
print(f"Checkpoints to average: {steps}")
print(f"Number of paths: {len(ckpt_paths)}")

Checkpoints to average: [87500, 100000, 112500, 125000, 137500]
Number of paths: 5


In [21]:
# Filter and load checkpoints
state_dicts = [torch.load(str(p), map_location='cpu') for p in ckpt_paths if p.exists()]
print(f"Loaded {len(state_dicts)} checkpoints")

Loaded 5 checkpoints


In [22]:
teacher_dicts = [sd["teacher"] for sd in state_dicts]

In [23]:
print(len(teacher_dicts), teacher_dicts[0])

5 OrderedDict([('backbone.cls_token', tensor([[[-0.0217, -0.0006, -0.0007,  ..., -0.0026, -0.0205, -0.0012]]])), ('backbone.pos_embed', tensor([[[-2.1716e-02, -5.7623e-04, -6.6399e-04,  ..., -2.5650e-03,
          -2.0453e-02, -1.1670e-03],
         [ 7.4053e-02, -1.6015e-02, -4.9749e-02,  ...,  5.1781e-03,
          -5.1990e-02,  2.9696e-02],
         [ 1.0096e-02, -2.5204e-03,  1.7941e-02,  ...,  5.7634e-03,
           2.1045e-02, -3.8860e-03],
         ...,
         [ 2.7285e-03,  1.5563e-03,  3.0562e-03,  ...,  7.9223e-03,
          -4.6409e-05, -1.1662e-04],
         [-6.7615e-04,  4.3785e-03,  6.6526e-03,  ...,  9.3127e-03,
          -1.0446e-02, -1.4434e-03],
         [-8.7385e-03,  1.6776e-02, -1.4831e-02,  ...,  9.8811e-03,
          -1.3756e-02,  1.3482e-02]]])), ('backbone.register_tokens', tensor([[[ 2.5682e-03, -8.4069e-05, -1.5675e-03,  ..., -3.9679e-04,
          -4.3028e-02, -1.2491e-03],
         [-1.7890e-01, -3.3005e-04, -1.0071e-02,  ..., -2.7869e-03,
           3.1

In [24]:
averaged_state_dict = {}
for key in teacher_dicts[0].keys():
    averaged_state_dict[key] = sum(td[key] for td in teacher_dicts) / len(teacher_dicts)

In [25]:
# Wrap back in the original structure
output_dict = {'teacher': averaged_state_dict}
print(f"Average dtype: {output_dict['teacher']['backbone.cls_token'].dtype}")

Average dtype: torch.float32


In [26]:
# Save averaged checkpoint
output_dir = Path("/data/ratna/aug/blur_weak/eval/averaged_87500_to_137500") # RSG - change this for a new range
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "teacher_checkpoint.pth"
torch.save(output_dict, output_path)
print(f"Saved to: {output_path}")

Saved to: /data/ratna/aug/blur_weak/eval/averaged_87500_to_137500/teacher_checkpoint.pth


The End

In [27]:
ckpt_path = Path("/data/ratna/aug/blur_weak/eval/averaged_87500_to_137500/teacher_checkpoint.pth")
state_dict = torch.load(ckpt_path, map_location="cpu")
print(state_dict)

{'teacher': {'backbone.cls_token': tensor([[[-0.0210, -0.0009, -0.0006,  ..., -0.0027, -0.0185, -0.0012]]]), 'backbone.pos_embed': tensor([[[-0.0209, -0.0009, -0.0006,  ..., -0.0027, -0.0185, -0.0012],
         [ 0.0672, -0.0140, -0.0465,  ...,  0.0032, -0.0469,  0.0261],
         [ 0.0084,  0.0038,  0.0172,  ...,  0.0043,  0.0194, -0.0026],
         ...,
         [ 0.0025,  0.0063,  0.0039,  ...,  0.0078, -0.0008, -0.0003],
         [ 0.0008,  0.0066,  0.0066,  ...,  0.0101, -0.0083, -0.0008],
         [-0.0105,  0.0144, -0.0131,  ...,  0.0087, -0.0170,  0.0119]]]), 'backbone.register_tokens': tensor([[[ 1.7688e-03, -2.5297e-04, -9.4794e-05,  ..., -1.5778e-03,
          -3.6307e-02, -8.5820e-04],
         [-1.6739e-01,  4.7714e-04, -9.8380e-03,  ..., -1.5371e-03,
           2.4949e-02,  1.7781e-03],
         [-5.7710e-03, -1.9457e-03, -7.5504e-02,  ..., -1.9224e-03,
           2.5956e-02,  1.4478e-03],
         [ 2.6776e-02,  6.0338e-04,  1.4072e-03,  ..., -1.0276e-04,
          -8.14